In [4]:
import numpy as np
import pandas as pd

In [5]:
#importation fichier .parquet
httrain= pd.read_parquet("history_toulouse_train.parquet").copy()
httest=pd.read_parquet("history_toulouse_test.parquet").copy()

In [8]:
# Création ensemble d'apprentissage
# Suppression des 8 dernières mesures de chaque station
rang_depuis_la_fin = (
    httrain
    .groupby(level="station")
    .cumcount(ascending=False)
)
rang_depuis_la_fin

station                   commit_at          
00001 - POIDS DE L'HUILE  2023-08-05 12:15:00    104774
                          2023-08-05 12:30:00    104773
                          2023-08-05 12:45:00    104772
                          2023-08-05 13:00:00    104771
                          2023-08-05 13:15:00    104770
                                                  ...  
TEST BEE                  2026-02-12 15:00:00         4
                          2026-02-12 15:15:00         3
                          2026-02-12 15:30:00         2
                          2026-02-12 15:45:00         1
                          2026-02-12 16:00:00         0
Length: 40012622, dtype: int64

In [12]:
history_toulouse_apprentissage = httrain[
    rang_depuis_la_fin >= 8
].copy()
history_toulouse_apprentissage

bikes  stands
station                  commit_at                         
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9
                         2023-08-05 12:30:00     10       9
                         2023-08-05 12:45:00     10       9
                         2023-08-05 13:00:00     11       8
                         2023-08-05 13:15:00     11       8
...                                             ...     ...
TEST BEE                 2026-02-12 13:00:00      0       3
                         2026-02-12 13:15:00      0       3
                         2026-02-12 13:30:00      0       3
                         2026-02-12 13:45:00      0       3
                         2026-02-12 14:00:00      0       3

[40006230 rows x 2 columns]

In [13]:
# Récupèration de la dernière date de mesure pour chaque station
# des données d'apprentissage
last_apprentissage = (
    history_toulouse_apprentissage
    .groupby(level="station")
    .tail(1)
    .copy()
)

last_apprentissage

,,bikes,stands
station,commit_at,,
00001 - POIDS DE L'HUILE,2026-07-31 19:45:00,6,10
00002 - LAFAYETTE,2026-07-31 21:45:00,5,13
00003 - ALSACE - LORRAINE - POMME,2025-11-10 09:45:00,7,11
00003 - ALSACE-LORRAINE - POMME,2026-07-31 19:45:00,15,1
00003 - POMME,2025-01-05 03:45:00,17,0
...,...,...,...
MONT LOUIS –AV. DE TOULOUSE,2026-03-23 18:15:00,10,5
NARBONNE - CLOTASSES,2025-08-22 17:15:00,0,16
PASTEUR - LAVOISIER,2025-08-05 08:15:00,0,0


In [18]:
# Creation de l'ensemble de validation
# On récupère les 9 dernières mesures :
# - la 1ère = dernière mesure dans l'ensemble d'apprentissage
# - les 8 suivantes = valeurs à prédire
last_9 = (
    httrain
    .groupby(level="station")
    .tail(9)
    .copy()
)
last_9

bikes  stands
station                  commit_at                         
00001 - POIDS DE L'HUILE 2026-07-31 19:45:00      6      10
                         2026-07-31 20:00:00      1      15
                         2026-07-31 20:15:00      1      15
                         2026-07-31 20:30:00      1      15
                         2026-07-31 20:45:00      1      15
...                                             ...     ...
TEST BEE                 2026-02-12 15:00:00      0       3
                         2026-02-12 15:15:00      0       3
                         2026-02-12 15:30:00      0       3
                         2026-02-12 15:45:00      0       3
                         2026-02-12 16:00:00      0       3

[7189 rows x 2 columns]

In [19]:
# On remet l'index sous forme de colonnes
last_9 = last_9.reset_index()
last_9

,station,commit_at,bikes,stands
0,00001 - POIDS DE L'HUILE,2026-07-31 19:45:00,6,10
1,00001 - POIDS DE L'HUILE,2026-07-31 20:00:00,1,15
2,00001 - POIDS DE L'HUILE,2026-07-31 20:15:00,1,15
3,00001 - POIDS DE L'HUILE,2026-07-31 20:30:00,1,15
4,00001 - POIDS DE L'HUILE,2026-07-31 20:45:00,1,15
...,...,...,...,...
7184,TEST BEE,2026-02-12 15:00:00,0,3
7185,TEST BEE,2026-02-12 15:15:00,0,3
7186,TEST BEE,2026-02-12 15:30:00,0,3
7187,TEST BEE,2026-02-12 15:45:00,0,3
